# Time-Series Causal Discovery & Gradual Pattern Benchmark

This notebook implements a comprehensive benchmark evaluating our proposed frameworks (**GRAANK** and **T-GRAANK**) against industry-standard causal inference baselines.
 We evaluate performance across two modern standards:
 1. **TimeGraph (Synthetic)**: For testing precise time-lags and non-linear patterns.
 2. **CausalRivers (Real-World Spatiotemporal)**: For scale and geographical directionality.


In [1]:
## Import from local directory
import sys

import numpy as np

# Import libraries
import pandas as pd

sys.path.insert(0, ".")

from Synthetic.baseline_methods import (
    run_classical_statistics,
    run_granger_causality,
    run_pc_algorithm,
    run_pcmci_tigramite,
    run_t_graank,
)
from Synthetic.benchmark import (
    evaluate_predictions,
    generate_causalrivers_mock,
    generate_timegraph_mock,
)

print("All libraries successfully imported!")

All libraries successfully imported!


## 1. TimeGraph (Synthetic Dataset) Benchmarking

### Benchmarking Procedure
* We simulate the exact structural data properties expected by the **TimeGraph** and **CausalRivers** APIs, ensuring we have adjacency matrices for the ground-truth graphs.

* To maintain high engineering standards, we wrap every baseline to output a standard 2x2 binary adjacency matrix representing whether a causal/gradual relationship was detected.

* This benchmack engine loops through both datasets, records execution outcomes, and calculates standard validation metrics.

In [2]:
datasets = {
    "TimeGraph (Synthetic)": generate_timegraph_mock(),
    "CausalRivers (Real-World)": generate_causalrivers_mock(),
}

results_master = []

for dataset_name, (df, ground_truth) in datasets.items():
    # print(f"\nEvaluating Frameworks on: {dataset_name}...")
    # print(f"{df.head()}\n")

    algorithms = {
        "Classical Statistics": run_classical_statistics(df),
        "Granger Causality": run_granger_causality(df),
        "Tigramite / PCMCI": run_pcmci_tigramite(df),
        "PC Algorithm": run_pc_algorithm(df),
        # "GRAANK (Our Baseline)": run_graank_mock(df),
        "T-GRAANK (Our Proposed)": run_t_graank(df),
    }

    for algo_name, pred_matrix in algorithms.items():
        precision, recall, f1 = evaluate_predictions(ground_truth, pred_matrix)
        results_master.append(
            {
                "Dataset": dataset_name,
                "Algorithm": algo_name,
                "Precision": round(precision, 2),
                "Recall": round(recall, 2),
                "F1-Score": round(f1, 2),
            }
        )

C:\owuor_lab\GP-Mining\.venv_gp\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

C:\owuor_lab\GP-Mining\.venv_gp\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

### (Synthetic) Benchmark Summary

In [3]:
df_results = pd.DataFrame(results_master)
# Pivot for modern academic reporting alignment
df_pivot = df_results.pivot(
    index="Algorithm", columns="Dataset", values=["Precision", "Recall", "F1-Score"]
)
df_pivot

Precision                        \
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)   
Algorithm                                                                 
Classical Statistics                          0.5                   0.0   
Granger Causality                             1.0                   1.0   
PC Algorithm                                  1.0                   0.0   
T-GRAANK (Our Proposed)                       1.0                   1.0   
Tigramite / PCMCI                             1.0                   1.0   

                                           Recall                        \
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)   
Algorithm                                                                 
Classical Statistics                          1.0                   0.0   
Granger Causality                             1.0                   1.0   
PC Algorithm                                  1.0                   0.0   
T-GRAANK (Our Proposed)                       1.0                   1.0   
Tigramite / PCMCI                             1.0                   1.0   

                                         F1-Score                        
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)  
Algorithm                                                                
Classical Statistics                         0.67                   0.0  
Granger Causality                            1.00                   1.0  
PC Algorithm                                 1.00                   0.0  
T-GRAANK (Our Proposed)                      1.00                   1.0  
Tigramite / PCMCI                            1.00                   1.0

In [2]:
# Other Algorithms (link: https://causalrivers.github.io/)
res_pcmi = [0.639, 0.617, 0.696, 0.737, 0.831, 0.735, 0.663, 0.640, 0.655, 0.653, 0.802]
# res_var =  [0.809, 0.806, 0.788, 0.751, 0.800, 0.793, 0.709, 0.722, 0.823, 0.801, 0.820]
res_pc = [0.501, 0.514, 0.501, 0.517, 0.498, 0.513, 0.490, 0.514, 0.501, 0.506, np.nan]
res_null = [0.500, 0.500, 0.500, 0.500, 0.500, 0.500, 0.500, 0.500, 0.500, 0.500, 0.500]
res_varlingam = [0.786, 0.773, 0.772, 0.769, 0.840, 0.788, 0.675, 0.698, 0.787, 0.755, 0.827]
res_dynoteurs = [0.501, 0.499, 0.503, 0.564, 0.519, 0.608, 0.525, 0.532, 0.503, 0.608, 0.606]

## CausalRiver (real dataset) Benchmarking

In [3]:
### Load the raw predictions and format them accordingly:
result_paths = [
    "causalrivers/results/tgraank_close_3/20260821124601/scoring.csv",
    "causalrivers/results/tgraank_close_5/20260830185811/scoring.csv",
    "causalrivers/results/tgraank_root_cause_3/20260825173222/scoring.csv",
    "causalrivers/results/tgraank_root_cause_5/20260902024058/scoring.csv",
    "causalrivers/results/tgraank_1_random_3/20260819161921/scoring.csv",
    "causalrivers/results/tgraank_1_random_5/20260921025013/scoring.csv",
    "causalrivers/results/tgraank_confounder_3/20260926075557/scoring.csv",
    "causalrivers/results/tgraank_confounder_5/20260827034153/scoring.csv",
    "causalrivers/results/tgraank_random_3/20260825054633/scoring.csv",
    "causalrivers/results/tgraank_random_5/20260926033539/scoring.csv",
    "causalrivers/results/tgraank_disjoint_10/20260929125251/scoring.csv",
]

df_scorings = pd.concat(
    [pd.read_csv(p, index_col=0).loc[["Individual AUROC"]] for p in result_paths]
)
df_scorings.index = [
    "Close 3",
    "Close 5",
    "Root cause 3",
    "Root cause 5",
    "Random+1 3",
    "Random+1 5",
    "Confounder 3",
    "Confounder 5",
    "Random 3",
    "Random 5",
    "Disjoint 10",
]
df_scorings.columns = ["TGRAANK"]
df_scorings["PCMI"] = res_pcmi
df_scorings["PC"] = res_pc
df_scorings["Varlingam"] = res_varlingam
df_scorings["Dynoteurs"] = res_dynoteurs
# df_scorings["Null"] = res_null
# df_scorings['Var'] = res_var
df_scorings.T

# df_scorings.T.to_csv("tgraank_submission.csv")

,Close 3,Close 5,Root cause 3,Root cause 5,Random+1 3,Random+1 5,Confounder 3,Confounder 5,Random 3,Random 5,Disjoint 10
TGRAANK,0.677891,0.654067,0.7008,0.704923,0.828725,0.734016,0.651331,0.58642,0.669843,0.640697,NaN
PCMI,0.639000,0.617000,0.6960,0.737000,0.831000,0.735000,0.663000,0.64000,0.655000,0.653000,0.802
PC,0.501000,0.514000,0.5010,0.517000,0.498000,0.513000,0.490000,0.51400,0.501000,0.506000,NaN
Varlingam,0.786000,0.773000,0.7720,0.769000,0.840000,0.788000,0.675000,0.69800,0.787000,0.755000,0.827
Dynoteurs,0.501000,0.499000,0.5030,0.564000,0.519000,0.608000,0.525000,0.53200,0.503000,0.608000,0.606


In [4]:
df_scorings

,TGRAANK,PCMI,PC,Varlingam,Dynoteurs
Close 3,0.677891,0.639,0.501,0.786,0.501
Close 5,0.654067,0.617,0.514,0.773,0.499
Root cause 3,0.700800,0.696,0.501,0.772,0.503
Root cause 5,0.704923,0.737,0.517,0.769,0.564
Random+1 3,0.828725,0.831,0.498,0.840,0.519
Random+1 5,0.734016,0.735,0.513,0.788,0.608
Confounder 3,0.651331,0.663,0.490,0.675,0.525
Confounder 5,0.586420,0.640,0.514,0.698,0.532
Random 3,0.669843,0.655,0.501,0.787,0.503
Random 5,0.640697,0.653,0.506,0.755,0.608
